## Traitement des données climatiques
Sur la base d'un choix de simulations CH2025

### Structure générale

1. Exploration et découpage des couches par le Canton de Vaud


## 1. Exploration et découpage des couches par le Canton de Vaud

In [ ]:
# exploration des fichiers de simulations

import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

# ============
# CONFIGURATION
# ============
filepath = r"../data/raw/MeteoCH/clmcom-cclm4-ecearth/ogd-climate-scenarios-ch2025-grid_ch_pr_clmcom-cclm4-ecearth_gwl2.0.nc"
varname = "pr"                                 # vérifier le nom exact dans ton fichier

# ============
# OUVERTURE DU FICHIER EN MODE LAZY
# ============
ds = xr.open_dataset(filepath, chunks={"time": 30})  # chunks pour dask (optimisation)
print("===== Dataset structure =====")
print(ds)

# Liste des variables
print("\n===== Variables =====")
print(list(ds.data_vars))

# Vérifier le nom exact de la variable si nécessaire
if varname not in ds.data_vars:
    raise ValueError(f"La variable '{varname}' n'est pas présente. Variables disponibles : {list(ds.data_vars)}")

da = ds[varname]

# ============
# INFORMATIONS TEMPORELLES
# ============
print("\n===== Time axis =====")
print("Start :", np.array(ds.time.values[0]).astype('datetime64[D]'))
print("End   :", np.array(ds.time.values[-1]).astype('datetime64[D]'))
print("Number of timesteps :", ds.time.size)

# ============
# APERÇU STATISTIQUE RAPIDE
# ============
print("\n===== Quick stats (lazy) =====")
print(da.mean(dim=["N", "E"]).compute())  # moyenne spatiale au cours du temps
print(da.isel(time=0).mean().compute())       # moyenne spatiale du premier jour

# ============
# VISUALISATION D’UNE CARTE
# ============
# (évite de calculer tout : affiche uniquement un pas de temps)
t_index = 0  # première date, tu peux changer 100, 365, etc.

print(f"\nAffichage de la carte pour time index = {t_index}")

plt.figure(figsize=(8,6))
da.isel(time=t_index).plot(cmap="viridis")
plt.title(f"{varname} – time index {t_index}")
plt.tight_layout()
plt.show()

# ============
# DISTRIBUTION DES VALEURS SUR UNE PÉRIODE COURTE
# ============
# utile pour vérifier la plage dynamique
subset = da.isel(time=slice(0, 30))  # premier mois approx
print("\nValeurs min/max pour 1 mois :")
print("min:", float(subset.min().compute()))
print("max:", float(subset.max().compute()))


In [ ]:
# découpage NetCDF CH2025 avec frontière canton de Vaud

import xarray as xr
import geopandas as gpd
import rioxarray
import os

# =========================================================
# PARAMÈTRES
# =========================================================
nc_file = r"../data/raw/MeteoCH/clmcom-cclm4-ecearth/ogd-climate-scenarios-ch2025-grid_ch_pr_clmcom-cclm4-ecearth_gwl2.0.nc"
vector_file = "../data/raw/Limite Canton Vaud/limites_officielles/OIT.MN95_OIT_TPR_LAD_MO_CANTON.shp"
output_file = "../data/processed/Climat/noresm_gwl2_pr_vaud.nc"

# ======================================================
# 1. Charger le NetCDF
# ======================================================
ds = xr.open_dataset(nc_file)

varname = list(ds.data_vars)[0]
da = ds[varname]

print("Dimensions brutes :", da.dims)

# ======================================================
# 2. Harmonisation des dimensions spatiales
# ======================================================
# Détection des noms possibles
spatial_dims = [d for d in da.dims if d.lower() in ["n", "e", "x", "y", "lon", "lat"]]

if "N" in da.dims and "E" in da.dims:
    da = da.rename({"N": "y", "E": "x"})
elif "rlat" in da.dims and "rlon" in da.dims:
    da = da.rename({"rlat": "y", "rlon": "x"})
elif "lat" in da.dims and "lon" in da.dims:
    da = da.rename({"lat": "y", "lon": "x"})
# Sinon, afficher message
print("Dimensions après renommage :", da.dims)

# Déclarer les dimensions spatiales
da = da.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)

# Écrire le CRS LV95 (EPSG:2056)
da = da.rio.write_crs("EPSG:2056", inplace=False)

# ======================================================
# 3. Charger la frontière vaudoise
# ======================================================
gdf = gpd.read_file(vector_file)

if gdf.crs.to_string() != "EPSG:2056":
    gdf = gdf.to_crs("EPSG:2056")

geom = gdf.geometry.values[0]

# ======================================================
# 4. Découpage spatial
# ======================================================
clipped = da.rio.clip([geom], gdf.crs)

# ======================================================
# 5. Sauvegarde
# ======================================================
clipped.to_netcdf(output_file)

print("Découpage terminé :", output_file)


## 2. Agrégation mensuelle et année climatique type

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. OUVERTURE DU FICHIER .nc
# ============================================================

#file_path = r"../data/raw/MeteoCH/clmcom-cclm4-ecearth/ogd-climate-scenarios-ch2025-grid_ch_pr_clmcom-cclm4-ecearth_gwl2.0.nc"
file_path = r"../data/processed/Climat/noresm_gwl2_tas_vaud.nc"
ds = xr.open_dataset(file_path)

print("===== Dataset structure =====")
print(ds)

# ============================================================
# 2. EXTRACTION DE LA VARIABLE (ici: tas)
# ============================================================

varname = "tas"   # changer à pr / tas / tasmax / tasmin selon le fichier
da = ds[varname]

print("\n===== Variable information =====")
print(da)

# Dimensions exactes: (time: 10950, N: 240, E: 370)
# Lon/Lat grilles 2D déjà présentes dans ds.lon et ds.lat

# ============================================================
# 3. AFFICHER UNE CARTE POUR UN INDEX TEMPOREL
# ============================================================

time_index = 0

plt.figure(figsize=(8, 6))
plt.imshow(
    da.isel(time=time_index),
    origin="lower",
    extent=[float(ds.E.min()), float(ds.E.max()),
            float(ds.N.min()), float(ds.N.max())]
)
plt.title(f"{varname} - time index {time_index}")
plt.colorbar(label=varname)
plt.xlabel("E (LV95)")
plt.ylabel("N (LV95)")
plt.show()

# ============================================================
# 4. CALCULS STATISTIQUES DE BASE
# ============================================================

stats = {
    "min": float(da.min()),
    "max": float(da.max()),
    "mean": float(da.mean()),
    "std": float(da.std()),
}

print("\n===== Global statistics on the full dataset =====")
for k, v in stats.items():
    print(f"{k}: {v}")

# ============================================================
# 5. EXTRACTIONS TEMPORELLES: ANNUELLES, SAISONNIÈRES, MENSUELLES
# ============================================================

# Convertir en Dataset temporel pour resampling
da_time = da.copy()
da_time["time"] = xr.decode_cf(ds).time

# **Moyennes annuelles**
annual_mean = da_time.resample(time="1Y").mean()

# **Moyennes mensuelles**
monthly_mean = da_time.resample(time="1M").mean()

# **Moyennes saisonnières (DJF, MAM, JJA, SON)**
seasonal_mean = da_time.groupby("time.season").mean()

print("\n===== Temporal summaries created =====")
print("annual_mean dims:", annual_mean.dims)
print("monthly_mean dims:", monthly_mean.dims)
print("seasonal_mean dims:", seasonal_mean.dims)

# Exemple d’affichage : carte de la moyenne annuelle de 0001
plt.figure(figsize=(8, 6))
annual_mean.isel(time=0).plot()
plt.title(f"{varname} - Annual mean (year index 0)")
plt.show()


In [ ]:
# Découpage du canton de Vaud
import xarray as xr
import rioxarray
import geopandas as gpd



# ============================================================
# 1. Charger le NetCDF et la variable
# ============================================================

file_path = r"../data/raw/MeteoCH/clmcom-cclm4-ecearth/ogd-climate-scenarios-ch2025-grid_ch_pr_clmcom-cclm4-ecearth_gwl2.0.nc"
varname = "pr"   # pr, tas, tasmax, tasmin

ds = xr.open_dataset(file_path)
da = ds[varname]

# Xarray → Rioxarray (indispensable pour clip)
da = da.rio.write_crs("EPSG:2056", inplace=True)

print("DataArray chargé avec CRS:", da.rio.crs)

# ============================================================
# 2. Charger le shapefile du canton de Vaud
# ============================================================

shp_path = "../data/raw/Limite Canton Vaud/limites_officielles/OIT.MN95_OIT_TPR_LAD_MO_CANTON.shp"
gdf = gpd.read_file(shp_path)
# Ajouter un buffer d’1 km autour de la géométrie
gdf_buffered = gdf.copy()
gdf_buffered["geometry"] = gdf_buffered.geometry.buffer(1000)  # 1000 m = 1 km
da_vaud = da.rio.clip(gdf_buffered.geometry, gdf_buffered.crs, drop=True)

print("Shapefile CRS:", gdf.crs)

# Reprojection si nécessaire
if gdf_buffered.crs != "EPSG:2056":
    gdf_buffered = gdf_buffered.to_crs("EPSG:2056")

# ============================================================
# 3. Découpage spatial (clip)
# ============================================================

da_vaud = da.rio.clip(gdf_buffered.geometry, gdf_buffered.crs, drop=True)

print("\n===== Découpage effectué =====")
print(da_vaud)

# ============================================================
# 4. Sauvegarde optionnelle (NetCDF)
# ============================================================

output_path = "../data/processed/Climat/noresm_gwl2_pr_vaud1.nc"
da_vaud.to_netcdf(output_path)

print(f"\nFichier découpé sauvegardé sous : {output_path}")



In [ ]:
# file_path = r"../data/raw/MeteoCH/clmcom-cclm4-ecearth/ogd-climate-scenarios-ch2025-grid_ch_pr_clmcom-cclm4-ecearth_gwl2.0.nc"
# varname = "pr"   # pr, tas, tasmax, tasmin
# shp_path = "../data/raw/Limite Canton Vaud/limites_officielles/OIT.MN95_OIT_TPR_LAD_MO_CANTON.shp"
def decoupe_vaud_netcdf(file_path, varname, output_path, shp_path= "../data/raw/Limite Canton Vaud/limites_officielles/OIT.MN95_OIT_TPR_LAD_MO_CANTON.shp"):
    # Charger le NetCDF et la variable
    ds = xr.open_dataset(file_path)
    da = ds[varname]
    da = da.rio.write_crs("EPSG:2056", inplace=True)

    # Charger le shapefile du canton de Vaud
    gdf = gpd.read_file(shp_path)
    gdf_buffered = gdf.copy()
    gdf_buffered["geometry"] = gdf_buffered.geometry.buffer(1000)  # 1 km buffer

    if gdf_buffered.crs != "EPSG:2056":
        gdf_buffered = gdf_buffered.to_crs("EPSG:2056")

    # Découpage spatial (clip)
    da_vaud = da.rio.clip(gdf_buffered.geometry, gdf_buffered.crs, drop=True)

    # Sauvegarde optionnelle (NetCDF)
    da_vaud.to_netcdf(output_path)

    print(f"Fichier découpé sauvegardé sous : {output_path}")

In [ ]:
#Découpage du canton de Vaud
import xarray as xr
import rioxarray
import geopandas as gpd

# ============================================================
# Noresm
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/cnrm-aladin-noresm/ogd-climate-scenarios-ch2025-grid_ch_pr_cnrm-aladin-noresm_gwl2.0.nc",
    varname = "pr",
    output_path = "../data/processed/Climat/noresm_gwl2_pr_vaud.nc"
)
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/cnrm-aladin-noresm/ogd-climate-scenarios-ch2025-grid_ch_tas_cnrm-aladin-noresm_gwl2.0.nc",
    varname = "tas",
    output_path = "../data/processed/Climat/noresm_gwl2_tas_vaud.nc"
)
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/cnrm-aladin-noresm/ogd-climate-scenarios-ch2025-grid_ch_tasmax_cnrm-aladin-noresm_gwl2.0.nc",
    varname = "tasmax",
    output_path = "../data/processed/Climat/noresm_gwl2_tasmax_vaud.nc"
)
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/cnrm-aladin-noresm/ogd-climate-scenarios-ch2025-grid_ch_tasmin_cnrm-aladin-noresm_gwl2.0.nc",
    varname = "tasmin",
    output_path = "../data/processed/Climat/noresm_gwl2_tasmin_vaud.nc"
)

decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/cnrm-aladin-noresm/ogd-climate-scenarios-ch2025-grid_ch_pr_cnrm-aladin-noresm_ref91-20.nc",
    varname = "pr",
    output_path = "../data/processed/Climat/noresm_ref91_pr_vaud.nc"
)
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/cnrm-aladin-noresm/ogd-climate-scenarios-ch2025-grid_ch_tas_cnrm-aladin-noresm_ref91-20.nc",
    varname = "tas",
    output_path = "../data/processed/Climat/noresm_ref91_tas_vaud.nc"
)
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/cnrm-aladin-noresm/ogd-climate-scenarios-ch2025-grid_ch_tasmax_cnrm-aladin-noresm_ref91-20.nc",
    varname = "tasmax",
    output_path = "../data/processed/Climat/noresm_ref91_tasmax_vaud.nc"
)
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/cnrm-aladin-noresm/ogd-climate-scenarios-ch2025-grid_ch_tasmin_cnrm-aladin-noresm_ref91-20.nc",
    varname = "tasmin",
    output_path = "../data/processed/Climat/noresm_ref91_tasmin_vaud.nc"
) 
# ============================================================
# ECEarth
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/clmcom-cclm4-ecearth/ogd-climate-scenarios-ch2025-grid_ch_pr_clmcom-cclm4-ecearth_gwl2.0.nc",
    varname = "pr",
    output_path = "../data/processed/Climat/ecearth_gwl2_pr_vaud.nc"
)
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/clmcom-cclm4-ecearth/ogd-climate-scenarios-ch2025-grid_ch_tas_clmcom-cclm4-ecearth_gwl2.0.nc",
    varname = "tas",
    output_path = "../data/processed/Climat/ecearth_gwl2_tas_vaud.nc"
)
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/clmcom-cclm4-ecearth/ogd-climate-scenarios-ch2025-grid_ch_tasmax_clmcom-cclm4-ecearth_gwl2.0.nc",
    varname = "tasmax",
    output_path = "../data/processed/Climat/ecearth_gwl2_tasmax_vaud.nc"
)
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/clmcom-cclm4-ecearth/ogd-climate-scenarios-ch2025-grid_ch_tasmin_clmcom-cclm4-ecearth_gwl2.0.nc",
    varname = "tasmin",
    output_path = "../data/processed/Climat/ecearth_gwl2_tasmin_vaud.nc"
)
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/clmcom-cclm4-ecearth/ogd-climate-scenarios-ch2025-grid_ch_pr_clmcom-cclm4-ecearth_ref91-20.nc",
    varname = "pr",
    output_path = "../data/processed/Climat/ecearth_ref91_pr_vaud.nc"
)
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/clmcom-cclm4-ecearth/ogd-climate-scenarios-ch2025-grid_ch_tas_clmcom-cclm4-ecearth_ref91-20.nc",
    varname = "tas",
    output_path = "../data/processed/Climat/ecearth_ref91_tas_vaud.nc"
)
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/clmcom-cclm4-ecearth/ogd-climate-scenarios-ch2025-grid_ch_tasmax_clmcom-cclm4-ecearth_ref91-20.nc",
    varname = "tasmax",
    output_path = "../data/processed/Climat/ecearth_ref91_tasmax_vaud.nc"
)
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/clmcom-cclm4-ecearth/ogd-climate-scenarios-ch2025-grid_ch_tasmin_clmcom-cclm4-ecearth_ref91-20.nc",
    varname = "tasmin",
    output_path = "../data/processed/Climat/ecearth_ref91_tasmin_vaud.nc"
)
# ============================================================
# HadGEM
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/clmcom-cclm4-hadgem/ogd-climate-scenarios-ch2025-grid_ch_pr_clmcom-cclm4-hadgem_gwl2.0.nc",
    varname = "pr",
    output_path = "../data/processed/Climat/hadgem_gwl2_pr_vaud.nc"
)
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/clmcom-cclm4-hadgem/ogd-climate-scenarios-ch2025-grid_ch_tas_clmcom-cclm4-hadgem_gwl2.0.nc",
    varname = "tas",
    output_path = "../data/processed/Climat/hadgem_gwl2_tas_vaud.nc"
)
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/clmcom-cclm4-hadgem/ogd-climate-scenarios-ch2025-grid_ch_tasmax_clmcom-cclm4-hadgem_gwl2.0.nc",
    varname = "tasmax",
    output_path = "../data/processed/Climat/hadgem_gwl2_tasmax_vaud.nc"
)
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/clmcom-cclm4-hadgem/ogd-climate-scenarios-ch2025-grid_ch_tasmin_clmcom-cclm4-hadgem_gwl2.0.nc",
    varname = "tasmin",
    output_path = "../data/processed/Climat/hadgem_gwl2_tasmin_vaud.nc"
)
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/clmcom-cclm4-hadgem/ogd-climate-scenarios-ch2025-grid_ch_pr_clmcom-cclm4-hadgem_ref91-20.nc",
    varname = "pr",
    output_path = "../data/processed/Climat/hadgem_ref91_pr_vaud.nc"
)
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/clmcom-cclm4-hadgem/ogd-climate-scenarios-ch2025-grid_ch_tas_clmcom-cclm4-hadgem_ref91-20.nc",
    varname = "tas",
    output_path = "../data/processed/Climat/hadgem_ref91_tas_vaud.nc"
)
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/clmcom-cclm4-hadgem/ogd-climate-scenarios-ch2025-grid_ch_tasmax_clmcom-cclm4-hadgem_ref91-20.nc",
    varname = "tasmax",
    output_path = "../data/processed/Climat/hadgem_ref91_tasmax_vaud.nc"
)
decoupe_vaud_netcdf(
    file_path = r"../data/raw/MeteoCH/clmcom-cclm4-hadgem/ogd-climate-scenarios-ch2025-grid_ch_tasmin_clmcom-cclm4-hadgem_ref91-20.nc",
    varname = "tasmin",
    output_path = "../data/processed/Climat/hadgem_ref91_tasmin_vaud.nc"
)


In [ ]:
# agregation temporelle mensuelle
import xarray as xr
import rioxarray  # noqa
import os


def aggregation_mensuelle_netcdf(
    input_path,
    varname,
    output_path,
    methode="auto"
):
    """
    Agrégation mensuelle d'un NetCDF journalier CH2025.

    Parameters
    ----------
    input_path : str
        Chemin du NetCDF journalier (découpé Vaud).
    varname : str
        Nom de la variable climatique (ex: 'pr', 'tas').
    output_path : str
        Chemin du NetCDF mensuel de sortie.
    methode : str
        'auto', 'sum' ou 'mean'.
        - auto : somme pour pr, moyenne sinon
    """

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # =========================
    # 1. Ouverture lazy
    # =========================
    ds = xr.open_dataset(input_path, chunks={"time": 30})
    da = ds[varname]

    # =========================
    # 2. Méthode d'agrégation
    # =========================
    if methode == "auto":
        if varname.lower() in ["pr", "precipitation"]:
            methode = "sum"
        else:
            methode = "mean"

    # =========================
    # 3. Agrégation mensuelle
    # =========================
    if methode == "sum":
        da_month = da.resample(time="MS").sum(dim="time")
    elif methode == "mean":
        da_month = da.resample(time="MS").mean(dim="time")
    else:
        raise ValueError("methode doit être 'auto', 'sum' ou 'mean'")

    # =========================
    # 4. Nettoyage attributs
    # =========================
    da_month.attrs.update(da.attrs)
    da_month.attrs["aggregation"] = "monthly"
    da_month.attrs["aggregation_method"] = methode

    # =========================
    # 5. Sauvegarde
    # =========================
    da_month.to_netcdf(
        output_path,
        encoding={varname: {"zlib": True, "complevel": 4}}
    )

    ds.close()

    print(f"Agrégation mensuelle terminée : {output_path}")


In [ ]:
# Agrégations
# Noresm
aggregation_mensuelle_netcdf(
    input_path="../data/processed/Climat/noresm_gwl2_pr_vaud.nc",
    varname="pr",
    output_path="../data/processed/Climat/noresm_gwl2_pr_vaud_monthly.nc"
)

In [ ]:
#Check
ds_m = xr.open_dataset("../data/processed/Climat/noresm_gwl2_pr_vaud_monthly.nc")
print(ds_m)
ds_d = xr.open_dataset("../data/processed/Climat/noresm_gwl2_pr_vaud.nc")
print(ds_d)

In [ ]:
# Agrégation mensuelle avec Dask
import xarray as xr
import os
import numpy as np

def create_typical_climate_year_dask(
    file_path,
    varname,
    output_path,
    chunks_time=365  # taille de chunk sur la dimension temps (1 an)
):
    """
    Crée une année climatique type (12 mois) à partir
    d'une série journalière CH2025 (30 ans), en utilisant Dask.

    Paramètres
    ----------
    file_path : str
        Fichier NetCDF journalier (découpé Vaud)
    varname : str
        Variable climatique (pr, tas, tasmax, tasmin)
    output_path : str
        Fichier NetCDF de sortie (12 mois)
    chunks_time : int
        Nombre de jours par chunk sur la dimension temps
    """

    # =========================================================
    # 1. OUVERTURE DU FICHIER AVEC CHUNKING (DASK)
    # =========================================================
    ds = xr.open_dataset(file_path, chunks={"time": chunks_time})
    da = ds[varname]

    # =========================================================
    # 2. AGRÉGATION MENSUELLE (CLIMATOLOGIE)
    # =========================================================
    da_monthly = da.groupby("time.month").mean("time")  # calcul lazy

    # =========================================================
    # 3. CRÉATION D'UNE DIMENSION TEMPORELLE PROPRE
    # =========================================================
    months = np.arange(1, 13)
    da_monthly = da_monthly.assign_coords(time=("month", months)) \
                             .swap_dims({"month": "time"}) \
                             .drop_vars("month")

    da_monthly["time"].attrs = {
        "long_name": "Month of typical climatological year",
        "description": "1 = January, ..., 12 = December"
    }

    # =========================================================
    # 4. ATTRIBUTS
    # =========================================================
    da_monthly.attrs.update({
        "description": "Typical climatological year (30-year mean, CH2025)",
        "GWL": ds.attrs.get("GWL", "2.0"),
        "source": "MeteoSwiss CH2025",
        "spatial_resolution": "1 km",
        "temporal_aggregation": "monthly climatology"
    })

    # =========================================================
    # 5. SAUVEGARDE (calcul réel forcé ici)
    # =========================================================
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    da_monthly.to_netcdf(output_path, compute=True)

    print(f"Année climatique type sauvegardée : {output_path}")

    return da_monthly


In [ ]:
create_typical_climate_year_dask(
    file_path="../data/processed/Climat/noresm_gwl2_pr_vaud.nc",
    varname="pr",
    output_path="../data/processed/Climat/noresm_gwl2_pr_vaud_typical_year.nc"
)

In [ ]:
# traitement de l'ensemble des modèles
import os
import xarray as xr
from pathlib import Path
# =========================================================
# PARAMÈTRES
# =========================================================
INPUT_DIR = "../data/processed/Climat/"
OUTPUT_DIR = "../data/processed/Climat/typical_climate_year/"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Variables climatiques attendues
VALID_VARS = ["pr", "tas", "tasmax", "tasmin"]

# =========================================================
# BOUCLE PRINCIPALE
# =========================================================
nc_files = sorted(Path(INPUT_DIR).glob("*.nc"))

print(f"{len(nc_files)} fichiers NetCDF trouvés.\n")

for i, nc_file in enumerate(nc_files, 1):

    print(f"[{i}/{len(nc_files)}] Traitement : {nc_file.name}")

    # Ouverture légère pour détecter la variable
    with xr.open_dataset(nc_file) as ds:
        vars_in_file = list(ds.data_vars)

    # Identification de la variable climatique
    var_candidates = [v for v in vars_in_file if v in VALID_VARS]

    if len(var_candidates) != 1:
        print(f"  ⚠ Variable non identifiable dans {nc_file.name} → ignoré")
        print(f"    Variables trouvées : {vars_in_file}\n")
        continue

    varname = var_candidates[0]

    # Nom de sortie
    output_name = nc_file.stem + "_typical_year.nc"
    output_path = os.path.join(OUTPUT_DIR, output_name)

    # Calcul
    create_typical_climate_year_dask(
        file_path=str(nc_file),
        varname=varname,
        output_path=output_path
    )

    print(f"  ✔ Sauvé : {output_name}\n")

print("Traitement terminé.")

In [ ]:
# exploration d'un fichier NetCDF climatologique type
import xarray as xr
import matplotlib.pyplot as plt

def explore_typical_year_nc(file_path, month=1, varname=None):
    """
    Explorer un fichier NetCDF climatologique type (12 mois).
    
    Paramètres
    ----------
    file_path : str
        Chemin vers le fichier NetCDF.
    month : int, optionnel
        Mois à visualiser (1=Janvier, ..., 12=Décembre). Par défaut 1.
    varname : str, optionnel
        Nom de la variable à afficher. Si None, utilise la première variable du dataset.
    """
    
    # Ouverture du dataset
    ds = xr.open_dataset(file_path, chunks={"time": 12})  # lazy loading
    
    # Choix de la variable
    if varname is None:
        varname = list(ds.data_vars)[0]
    
    da = ds[varname]
    
    print("===== Informations du dataset =====")
    print(ds)
    print("\n===== Dimensions =====")
    print(da.dims)
    print("\n===== Coordonnées =====")
    print(da.coords)
    
    # Vérification du mois
    if month < 1 or month > 12:
        raise ValueError("month doit être entre 1 et 12")
    
    # Sélection du mois (index = month-1)
    da_month = da.isel(time=month-1)
    
    # Plot
    plt.figure(figsize=(8,6))
    da_month.plot(cmap="viridis")
    plt.title(f"{varname} - Mois {month}")
    plt.xlabel("E")
    plt.ylabel("N")
    plt.show()
    
    return da_month


In [ ]:
explore_typical_year_nc(
    file_path="../data/processed/Climat/typical_climate_year/ecearth_gwl2_pr_vaud_typical_year_mm_month.nc",
    month=8,)


## 3. Correction des précipitations (mm/jour → mm/mois)

In [ ]:
# Correction des précipitations

import xarray as xr
import numpy as np
import os


def convert_pr_mmday_to_mmmonth(
    input_nc,
    varname="pr",
    output_nc=None
):
    """
    Convertit une climatologie mensuelle de précipitations
    de mm/jour vers mm/mois, en multipliant chaque mois
    par son nombre de jours.

    Dimensions spatiales et temporelles inchangées.

    Paramètres
    ----------
    input_nc : str
        Fichier NetCDF d'entrée (typical year, 12 mois)
    varname : str
        Nom de la variable de précipitation (par défaut 'pr')
    output_nc : str, optionnel
        Fichier NetCDF de sortie. Si None, suffixe automatique.
    """

    # =========================================================
    # 1. OUVERTURE
    # =========================================================
    ds = xr.open_dataset(input_nc)
    da = ds[varname]

    if da.sizes.get("time", None) != 12:
        raise ValueError("La dimension temporelle n'a pas 12 mois.")

    # =========================================================
    # 2. NOMBRE DE JOURS PAR MOIS (calendrier standard)
    # =========================================================
    days_in_month = xr.DataArray(
        np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]),
        dims="time",
        coords={"time": da["time"]}
    )

    # =========================================================
    # 3. CONVERSION mm/jour → mm/mois
    # =========================================================
    da_mm_month = da * days_in_month

    # =========================================================
    # 4. ATTRIBUTS
    # =========================================================
    da_mm_month.attrs.update(da.attrs)
    da_mm_month.attrs.update({
        "units": "mm/month",
        "long_name": "Monthly precipitation sum (typical climatological year)",
        "processing": "Converted from mm/day to mm/month using month length",
    })

    # =========================================================
    # 5. DATASET DE SORTIE
    # =========================================================
    ds_out = da_mm_month.to_dataset(name=varname)
    ds_out.attrs.update(ds.attrs)
    ds_out.attrs["history"] = (
        ds.attrs.get("history", "") +
        " | Monthly precipitation converted from mm/day to mm/month"
    )

    # =========================================================
    # 6. SAUVEGARDE
    # =========================================================
    if output_nc is None:
        base, ext = os.path.splitext(input_nc)
        output_nc = base + "_mm_month.nc"

    os.makedirs(os.path.dirname(output_nc), exist_ok=True)
    ds_out.to_netcdf(output_nc)

    print(f"✔ Conversion effectuée : {output_nc}")

    return ds_out


In [ ]:
convert_pr_mmday_to_mmmonth(
    input_nc="../data/processed/Climat/typical_climate_year/hadgem_gwl2_pr_vaud_typical_year.nc",
    varname="pr",
    output_nc="../data/processed/Climat/typical_climate_year/hadgem_gwl2_pr_vaud_typical_year_mm_month.nc"
)

convert_pr_mmday_to_mmmonth(
    input_nc="../data/processed/Climat/typical_climate_year/hadgem_ref91_pr_vaud_typical_year.nc",
    varname="pr",
    output_nc="../data/processed/Climat/typical_climate_year/hadgem_ref91_pr_vaud_typical_year_mm_month.nc"
)
convert_pr_mmday_to_mmmonth(
    input_nc="../data/processed/Climat/typical_climate_year/ecearth_gwl2_pr_vaud_typical_year.nc",
    output_nc="../data/processed/Climat/typical_climate_year/ecearth_gwl2_pr_vaud_typical_year_mm_month.nc"
)
convert_pr_mmday_to_mmmonth(
    input_nc="../data/processed/Climat/typical_climate_year/ecearth_ref91_pr_vaud_typical_year.nc",
    output_nc="../data/processed/Climat/typical_climate_year/ecearth_ref91_pr_vaud_typical_year_mm_month.nc"
)
convert_pr_mmday_to_mmmonth(
    input_nc="../data/processed/Climat/typical_climate_year/noresm_gwl2_pr_vaud_typical_year.nc",
    output_nc="../data/processed/Climat/typical_climate_year/noresm_gwl2_pr_vaud_typical_year_mm_month.nc"
)
convert_pr_mmday_to_mmmonth(
    input_nc="../data/processed/Climat/typical_climate_year/noresm_ref91_pr_vaud_typical_year.nc",
    output_nc="../data/processed/Climat/typical_climate_year/noresm_ref91_pr_vaud_typical_year_mm_month.nc"
)


## 4. Agrégation saisonnière

In [ ]:
# Agrégration saisonnière des années typiques
import xarray as xr
import numpy as np
import os


def aggregate_to_seasons(
    input_nc,
    varname,
    output_nc=None
):
    """
    Agrège une année climatique type (12 mois) en 4 saisons.

    Paramètres
    ----------
    input_nc : str
        NetCDF d'entrée (12 mois)
    varname : str
        Variable climatique (pr, tas, tasmax, tasmin)
    output_nc : str, optionnel
        NetCDF de sortie (4 saisons)
    """

    ds = xr.open_dataset(input_nc)
    da = ds[varname]

    # Définition des saisons
    seasons = {
        "DJF": [12, 1, 2],
        "MAM": [3, 4, 5],
        "JJA": [6, 7, 8],
        "SON": [9, 10, 11]
    }

    seasonal_data = []

    for season, months in seasons.items():
        da_season = da.sel(time=months)

        if varname == "pr":
            da_agg = da_season.sum("time")
        else:
            da_agg = da_season.mean("time")

        da_agg = da_agg.expand_dims(time=[season])
        seasonal_data.append(da_agg)

    # Concaténation
    da_seasonal = xr.concat(seasonal_data, dim="time")

    # Attributs
    da_seasonal.attrs.update(da.attrs)
    da_seasonal.attrs.update({
        "temporal_aggregation": "seasonal",
        "seasons": "DJF, MAM, JJA, SON"
    })

    # Dataset de sortie
    ds_out = da_seasonal.to_dataset(name=varname)
    ds_out.attrs.update(ds.attrs)

    # Sauvegarde
    if output_nc is None:
        base, ext = os.path.splitext(input_nc)
        output_nc = base + "_seasonal.nc"

    os.makedirs(os.path.dirname(output_nc), exist_ok=True)
    ds_out.to_netcdf(output_nc)

    print(f"✔ Agrégation saisonnière effectuée : {output_nc}")

    return ds_out


In [ ]:
# Noresm
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/noresm_gwl2_tas_vaud_typical_year.nc",
    varname="tas",
    output_nc="../data/processed/Climat/typical_seasons/noresm_gwl2_tas_seasonal.nc",
)
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/noresm_ref91_tas_vaud_typical_year.nc",
    varname="tas",
    output_nc="../data/processed/Climat/typical_seasons/noresm_ref91_tas_seasonal.nc",
)
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/noresm_gwl2_tasmin_vaud_typical_year.nc",
    varname="tasmin",
    output_nc="../data/processed/Climat/typical_seasons/noresm_gwl2_tasmin_seasonal.nc",
)
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/noresm_ref91_tasmin_vaud_typical_year.nc",
    varname="tasmin",
    output_nc="../data/processed/Climat/typical_seasons/noresm_ref91_tasmin_seasonal.nc",
)
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/noresm_gwl2_tasmax_vaud_typical_year.nc",
    varname="tasmax",
    output_nc="../data/processed/Climat/typical_seasons/noresm_gwl2_tasmax_seasonal.nc",
)
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/noresm_ref91_tasmax_vaud_typical_year.nc",
    varname="tasmax",
    output_nc="../data/processed/Climat/typical_seasons/noresm_ref91_tasmax_seasonal.nc",
)
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/noresm_gwl2_pr_vaud_typical_year_mm_month.nc",
    varname="pr",
    output_nc="../data/processed/Climat/typical_seasons/noresm_gwl2_pr_seasonal.nc",
)
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/noresm_ref91_pr_vaud_typical_year_mm_month.nc",
    varname="pr",
    output_nc="../data/processed/Climat/typical_seasons/noresm_ref91_pr_seasonal.nc",
)
#============================================================
# ECEarth
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/ecearth_gwl2_tas_vaud_typical_year.nc",
    varname="tas",
    output_nc="../data/processed/Climat/typical_seasons/ecearth_gwl2_tas_seasonal.nc",
)
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/ecearth_ref91_tas_vaud_typical_year.nc",
    varname="tas",
    output_nc="../data/processed/Climat/typical_seasons/ecearth_ref91_tas_seasonal.nc",
)
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/ecearth_gwl2_tasmin_vaud_typical_year.nc",
    varname="tasmin",
    output_nc="../data/processed/Climat/typical_seasons/ecearth_gwl2_tasmin_seasonal.nc",
)
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/ecearth_ref91_tasmin_vaud_typical_year.nc",
    varname="tasmin",
    output_nc="../data/processed/Climat/typical_seasons/ecearth_ref91_tasmin_seasonal.nc",
)
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/ecearth_gwl2_tasmax_vaud_typical_year.nc",
    varname="tasmax",
    output_nc="../data/processed/Climat/typical_seasons/ecearth_gwl2_tasmax_seasonal.nc",
)
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/ecearth_ref91_tasmax_vaud_typical_year.nc",
    varname="tasmax",
    output_nc="../data/processed/Climat/typical_seasons/ecearth_ref91_tasmax_seasonal.nc",
)
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/ecearth_gwl2_pr_vaud_typical_year_mm_month.nc",
    varname="pr",
    output_nc="../data/processed/Climat/typical_seasons/ecearth_gwl2_pr_seasonal.nc",
)
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/ecearth_ref91_pr_vaud_typical_year_mm_month.nc",
    varname="pr",
    output_nc="../data/processed/Climat/typical_seasons/ecearth_ref91_pr_seasonal.nc",
)
#============================================================
# HadGEM
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/hadgem_gwl2_tas_vaud_typical_year.nc",
    varname="tas",
    output_nc="../data/processed/Climat/typical_seasons/hadgem_gwl2_tas_seasonal.nc",
)
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/hadgem_ref91_tas_vaud_typical_year.nc",
    varname="tas",
    output_nc="../data/processed/Climat/typical_seasons/hadgem_ref91_tas_seasonal.nc",
)
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/hadgem_gwl2_tasmin_vaud_typical_year.nc",
    varname="tasmin",
    output_nc="../data/processed/Climat/typical_seasons/hadgem_gwl2_tasmin_seasonal.nc",
)
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/hadgem_ref91_tasmin_vaud_typical_year.nc",
    varname="tasmin",
    output_nc="../data/processed/Climat/typical_seasons/hadgem_ref91_tasmin_seasonal.nc",
)
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/hadgem_gwl2_tasmax_vaud_typical_year.nc",
    varname="tasmax",
    output_nc="../data/processed/Climat/typical_seasons/hadgem_gwl2_tasmax_seasonal.nc",
)
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/hadgem_ref91_tasmax_vaud_typical_year.nc",
    varname="tasmax",
    output_nc="../data/processed/Climat/typical_seasons/hadgem_ref91_tasmax_seasonal.nc",
)
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/hadgem_gwl2_pr_vaud_typical_year_mm_month.nc",
    varname="pr",
    output_nc="../data/processed/Climat/typical_seasons/hadgem_gwl2_pr_seasonal.nc",
)
aggregate_to_seasons(
    input_nc="../data/processed/Climat/typical_climate_year/hadgem_ref91_pr_vaud_typical_year_mm_month.nc",
    varname="pr",
    output_nc="../data/processed/Climat/typical_seasons/hadgem_ref91_pr_seasonal.nc",
)

## 5. Calcul de l'indice de sécheresse

In [ ]:
# normalisation des variables
import xarray as xr
import numpy as np

def normalize_minmax(da, inverse=False):
    """
    Normalisation min-max spatiale.
    inverse=True pour les variables où une valeur élevée
    signifie plus de sécheresse (températures).
    """
    da_min = da.min(dim=("N", "E"))
    da_max = da.max(dim=("N", "E"))

    norm = (da - da_min) / (da_max - da_min)

    if inverse:
        norm = 1 - norm

    return norm
# sous-indice printanier
def spring_drought_index(ds_pr, ds_tas):
    """
    Sous-indice de sécheresse printanière (MAM)
    """
    pr = ds_pr["pr"].sel(time="MAM")
    tas = ds_tas["tas"].sel(time="MAM")

    pr_n = normalize_minmax(pr, inverse=True)
    tas_n = normalize_minmax(tas, inverse=False)

    I_spring = 0.7 * pr_n + 0.3 * tas_n

    I_spring.name = "I_spring_drought"
    I_spring.attrs["description"] = "Spring drought sensitivity index (MAM)"

    return I_spring

# sous-indice estival
def summer_drought_index(ds_pr, ds_tas, ds_tasmax):
    """
    Sous-indice de sécheresse estivale (JJA)
    """
    pr = ds_pr["pr"].sel(time="JJA")
    tas = ds_tas["tas"].sel(time="JJA")
    tasmax = ds_tasmax["tasmax"].sel(time="JJA")

    pr_n = normalize_minmax(pr, inverse=True)
    tas_n = normalize_minmax(tas, inverse=False)
    tasmax_n = normalize_minmax(tasmax, inverse=False)

    I_summer = (
        0.5 * pr_n +
        0.3 * tas_n +
        0.2 * tasmax_n
    )

    I_summer.name = "I_summer_drought"
    I_summer.attrs["description"] = "Summer drought sensitivity index (JJA)"

    return I_summer

# indice climatique sécheresse
def climate_drought_index(I_spring, I_summer):
    """
    Indice climatique final de sécheresse
    """
    I_climate = 0.3 * I_spring + 0.7 * I_summer

    I_climate.name = "I_climate_drought"
    I_climate.attrs.update({
        "description": "Composite climate drought sensitivity index",
        "spring_weight": 0.3,
        "summer_weight": 0.7
    })

    return I_climate


In [ ]:
# pipeline complète pour un modèle donné
def compute_climate_drought_index(ds_pr_path, ds_tas_path, ds_tasmax_path, output_path, model_name, gwl):
    """
    Calcul complet de l'indice climatique de sécheresse
    à partir des fichiers saisonniers.
    """
    # OUVERTURE DES DONNÉES
    ds_pr = xr.open_dataset(ds_pr_path, chunks={"N": 100, "E": 100})
    ds_tas = xr.open_dataset(ds_tas_path, chunks={"N": 100, "E": 100})
    ds_tasmax = xr.open_dataset(ds_tasmax_path, chunks={"N": 100, "E": 100})

    # CALCUL DES SOUS-INDICES
    I_spring = spring_drought_index(ds_pr, ds_tas)
    I_summer = summer_drought_index(ds_pr, ds_tas, ds_tasmax)

    # INDICE FINAL
    I_climate = climate_drought_index(I_spring, I_summer)

    # SAUVEGARDE
    I_climate.to_netcdf(
        f"{output_path}/I_climate_drought_{model_name}_{gwl}.nc"
    )
    I_spring.to_netcdf(
        f"{output_path}/I_spring_drought_{model_name}_{gwl}.nc"
    )
    I_summer.to_netcdf(
        f"{output_path}/I_summer_drought_{model_name}_{gwl}.nc"
    )
    print(f"Indices de sécheresse sauvegardés pour {model_name} {gwl} à :")
    
    return output_path


In [ ]:
# application pipeline à tous les modèles
compute_climate_drought_index(ds_pr_path="../data/processed/Climat/typical_seasons/noresm_gwl2_pr_seasonal.nc",
                              ds_tas_path="../data/processed/Climat/typical_seasons/noresm_gwl2_tas_seasonal.nc",
                              ds_tasmax_path="../data/processed/Climat/typical_seasons/noresm_gwl2_tasmax_seasonal.nc",
                              output_path="../data/processed/Climat/indices_secheresse",
                              model_name="noresm",
                              gwl="gwl2")
compute_climate_drought_index(ds_pr_path="../data/processed/Climat/typical_seasons/noresm_ref91_pr_seasonal.nc",
                              ds_tas_path="../data/processed/Climat/typical_seasons/noresm_ref91_tas_seasonal.nc",
                              ds_tasmax_path="../data/processed/Climat/typical_seasons/noresm_ref91_tasmax_seasonal.nc",
                              output_path="../data/processed/Climat/indices_secheresse",
                              model_name="noresm",
                              gwl="ref91")
compute_climate_drought_index(ds_pr_path="../data/processed/Climat/typical_seasons/ecearth_gwl2_pr_seasonal.nc",
                              ds_tas_path="../data/processed/Climat/typical_seasons/ecearth_gwl2_tas_seasonal.nc",
                              ds_tasmax_path="../data/processed/Climat/typical_seasons/ecearth_gwl2_tasmax_seasonal.nc",
                              output_path="../data/processed/Climat/indices_secheresse",
                              model_name="ecearth",
                              gwl="gwl2")
compute_climate_drought_index(ds_pr_path="../data/processed/Climat/typical_seasons/ecearth_ref91_pr_seasonal.nc",
                              ds_tas_path="../data/processed/Climat/typical_seasons/ecearth_ref91_tas_seasonal.nc",
                              ds_tasmax_path="../data/processed/Climat/typical_seasons/ecearth_ref91_tasmax_seasonal.nc",
                              output_path="../data/processed/Climat/indices_secheresse",
                              model_name="ecearth",
                              gwl="ref91")
compute_climate_drought_index(ds_pr_path="../data/processed/Climat/typical_seasons/hadgem_gwl2_pr_seasonal.nc",
                              ds_tas_path="../data/processed/Climat/typical_seasons/hadgem_gwl2_tas_seasonal.nc",
                              ds_tasmax_path="../data/processed/Climat/typical_seasons/hadgem_gwl2_tasmax_seasonal.nc",
                              output_path="../data/processed/Climat/indices_secheresse",
                              model_name="hadgem",
                              gwl="gwl2")
compute_climate_drought_index(ds_pr_path="../data/processed/Climat/typical_seasons/hadgem_ref91_pr_seasonal.nc",
                              ds_tas_path="../data/processed/Climat/typical_seasons/hadgem_ref91_tas_seasonal.nc",
                              ds_tasmax_path="../data/processed/Climat/typical_seasons/hadgem_ref91_tasmax_seasonal.nc",
                              output_path="../data/processed/Climat/indices_secheresse",
                              model_name="hadgem",
                              gwl="ref91")


In [ ]:
#fonction d'exploration des indices
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

def explore_climate_index(
    nc_path,
    vmin=0,
    vmax=1,
    cmap="YlOrRd",
    show_hist=True
):
    """
    Exploration visuelle et statistique d'un indice climatique final.

    Paramètres
    ----------
    nc_path : str
        Chemin vers le fichier NetCDF de l'indice
    vmin, vmax : float
        Bornes de visualisation
    cmap : str
        Colormap matplotlib
    show_hist : bool
        Affiche un histogramme des valeurs
    """

    # =========================================================
    # 1. OUVERTURE
    # =========================================================
    ds = xr.open_dataset(nc_path)

    # Détection automatique de la variable
    if len(ds.data_vars) != 1:
        raise ValueError(
            f"Le fichier contient plusieurs variables : {list(ds.data_vars)}"
        )

    varname = list(ds.data_vars)[0]
    da = ds[varname]

    print("=================================================")
    print("Exploration de l'indice climatique")
    print("-------------------------------------------------")
    print(f"Fichier      : {nc_path}")
    print(f"Variable     : {varname}")
    print(f"Dimensions   : {da.dims}")
    print(f"Shape        : {da.shape}")
    print(f"Unités       : {da.attrs.get('units', 'non spécifiées')}")
    print("-------------------------------------------------")

    # =========================================================
    # 2. STATISTIQUES GLOBALES
    # =========================================================
    da_vals = da.values

    print("Statistiques globales :")
    print(f"  Min  : {np.nanmin(da_vals):.3f}")
    print(f"  Max  : {np.nanmax(da_vals):.3f}")
    print(f"  Mean : {np.nanmean(da_vals):.3f}")
    print(f"  Std  : {np.nanstd(da_vals):.3f}")

    # =========================================================
    # 3. CARTE
    # =========================================================
    plt.figure(figsize=(7, 6))
    da.plot(
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        add_colorbar=True
    )
    plt.title(f"{varname} – indice climatique")
    plt.axis("equal")
    plt.tight_layout()
    plt.show()

    # =========================================================
    # 4. HISTOGRAMME (optionnel)
    # =========================================================
    if show_hist:
        plt.figure(figsize=(6, 4))
        plt.hist(
            da_vals.flatten(),
            bins=50,
            edgecolor="black"
        )
        plt.xlabel("Valeur de l'indice")
        plt.ylabel("Nombre de pixels")
        plt.title("Distribution spatiale de l'indice")
        plt.tight_layout()
        plt.show()

    ds.close()


In [ ]:
explore_climate_index(
    nc_path="../data/processed/Climat/indices_secheresse/I_climate_drought_hadgem_gwl2.nc",
    vmin=0,
    vmax=1,
    cmap="YlOrRd",
    show_hist=True
)

6. Export GeoTIFF et normalisation

In [ ]:
# conversion des NetCDF en Raster comparabilité avec autres indices
import xarray as xr
import rioxarray as rxr
import os

def export_all_indices_nc_to_geotiff(
    nc_path,
    output_dir,
    nodata_value=-9999.0
):
    """
    Exporte toutes les variables d'indice d'un NetCDF en GeoTIFF.
    """

    ds = xr.open_dataset(nc_path)

    os.makedirs(output_dir, exist_ok=True)

    for varname in ds.data_vars:

        da = ds[varname]

        # Ignorer variables non spatiales
        if not {"N", "E"}.issubset(set(da.dims)):
            continue

        print(f"  → Export de {varname}")

        # Sélection spatiale
        if "time" in da.dims:
            da = da.isel(time=0)

        # Déclaration spatiale
        da = da.rio.set_spatial_dims(
            x_dim="E",
            y_dim="N",
            inplace=False
        )
        da = da.rio.write_crs("EPSG:2056", inplace=False)

        # NoData
        da = da.where(~da.isnull(), nodata_value)
        da = da.rio.write_nodata(nodata_value)

        # Nom de sortie
        output_tif = os.path.join(
            output_dir,
            f"{os.path.splitext(os.path.basename(nc_path))[0]}.tif"
        )

        da.rio.to_raster(
            output_tif,
            driver="GTiff",
            compress="LZW",
            dtype="float32"
        )

        print(f"     ✔ {output_tif}")


In [ ]:
from pathlib import Path

INPUT_DIR = "../data/processed/Climat/indices_secheresse/"
OUTPUT_DIR = "../data/processed/Climat/indices_secheresse_tif2/"

nc_files = sorted(Path(INPUT_DIR).glob("*.nc"))

for nc_file in nc_files:
    print(f"\nTraitement : {nc_file.name}")
    export_all_indices_nc_to_geotiff(
        nc_path=str(nc_file),
        output_dir=OUTPUT_DIR
    )


In [ ]:
# renormalisation clip percentile
# afin que tout soit entre 0 et 1 et clippé comme pour les autres indices
from pathlib import Path

# =========================================================
# PARAMÈTRES
# =========================================================
INPUT_DIR = Path("../data/processed/Climat/indices_secheresse_tif")
OUTPUT_DIR = Path("../data/processed/Climat/indices_secheresse_tif_norm")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pmin = 2.0
pmax = 98.0
prelog = False
symmetric = False
overwrite = False

tif_files = sorted(INPUT_DIR.glob("*.tif"))

print(f"{len(tif_files)} fichiers TIF trouvés.\n")

for i, tif_path in enumerate(tif_files, 1):

    # ex: I_climat_noresm_ref91.tif → I_climat_noresm_ref91_norm.tif
    output_name = tif_path.stem + "_norm" + tif_path.suffix
    output_path = OUTPUT_DIR / output_name

    print(f"[{i}/{len(tif_files)}] Normalisation : {tif_path.name}")
    try:
        normalize_with_clip(
            input_path=tif_path,
            output_path=output_path,
            pmin=pmin,
            pmax=pmax,
            prelog=prelog,
            symmetric=symmetric,
            overwrite=overwrite
        )
        print("  ✔ OK\n")

    except Exception as e:
        print(f"  ✖ ERREUR sur {tif_path.name}")
        print(f"    {e}\n")

print("Normalisation terminée pour tous les indices.")


## 7. Agrégation inter-modèles (ensemble)

In [ ]:
#script d'agrégration inter-modèles
"""
architecture d'entrée
indices_secheresse_tif_norm/
├── ref91/
│   ├── noresm_ref91.tif
│   ├── ecearth_ref91.tif
│   └── hadgem_ref91.tif
├── gwl2/
│   ├── noresm_gwl2.tif
│   ├── ecearth_gwl2.tif
│   └── hadgem_gwl2.tif

architecture de sortie
indices_secheresse_ensemble/
├── I_climat_ref91_ensemble.tif
└── I_climat_gwl2_ensemble.tif
"""

import os
import json
import numpy as np
import rasterio

def aggregate_rasters_mean(input_files, output_path, *, overwrite=False):
    """
    Agrège plusieurs rasters (moyenne pixel à pixel).
    - Gère correctement les nodata
    - Suppose que tous les rasters sont alignés
    """

    if os.path.exists(output_path) and not overwrite:
        raise FileExistsError(f"{output_path} already exists. Use overwrite=True.")

    arrays = []
    nodata_values = []

    for fp in input_files:
        with rasterio.open(fp) as src:
            arr = src.read(1).astype(np.float32)
            profile = src.profile
            nodata = src.nodata

        if nodata is not None:
            arr = np.where(arr == nodata, np.nan, arr)

        arrays.append(arr)
        nodata_values.append(nodata)

    stack = np.stack(arrays, axis=0)  # shape (n_models, H, W)

    # moyenne en ignorant les NaN
    mean_arr = np.nanmean(stack, axis=0).astype(np.float32)

    # gestion nodata en sortie
    out_nodata = -9999
    mean_arr_out = np.where(np.isfinite(mean_arr), mean_arr, out_nodata)

    profile.update(
        dtype=rasterio.float32,
        nodata=out_nodata,
        compress="deflate",
        count=1
    )

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(mean_arr_out, 1)

    # métadonnées simples
    meta = {
        "aggregation": "mean",
        "n_models": len(input_files),
        "models": [os.path.basename(fp) for fp in input_files]
    }

    meta_path = os.path.splitext(output_path)[0] + "_meta.json"
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print(f"Saved ensemble raster: {output_path}")
    print(f"Saved metadata: {meta_path}")

    return output_path, meta_path


In [ ]:
#reférence ref91
input_ref91 = [
    "../data/processed/Climat/indices_secheresse_tif_norm/ref91/I_climate_drought_noresm_ref91_norm.tif",
    "../data/processed/Climat/indices_secheresse_tif_norm/ref91/I_climate_drought_ecearth_ref91_norm.tif",
    "../data/processed/Climat/indices_secheresse_tif_norm/ref91/I_climate_drought_hadgem_ref91_norm.tif",
]

aggregate_rasters_mean(
    input_ref91,
    "../data/processed/Climat/indices_secheresse_ensemble/I_climat_ref91_ensemble.tif",
    overwrite=True
)
# GWL2
input_gwl2 = [
    "../data/processed/Climat/indices_secheresse_tif_norm/gwl2/I_climate_drought_noresm_gwl2_norm.tif",
    "../data/processed/Climat/indices_secheresse_tif_norm/gwl2/I_climate_drought_ecearth_gwl2_norm.tif",
    "../data/processed/Climat/indices_secheresse_tif_norm/gwl2/I_climate_drought_hadgem_gwl2_norm.tif",
]

aggregate_rasters_mean(
    input_gwl2,
    "../data/processed/Climat/indices_secheresse_ensemble/I_climat_gwl2_ensemble.tif",
    overwrite=True
)


In [ ]:
import rasterio
import numpy as np

test_file = "../data/processed/Climat/indices_secheresse_ensemble/I_climat_ref91_ensemble.tif"

with rasterio.open(test_file) as src:
    arr = src.read(1)
    print("min:", np.nanmin(arr))
    print("max:", np.nanmax(arr))

## 8. Différence GWL2 − REF91

In [ ]:
# Différence entre GWL2 et ref91
import os
import numpy as np
import rasterio

def compute_raster_difference(
    raster_gwl2,
    raster_ref91,
    output_path,
    nodata_value=-9999.0,
    overwrite=False
):
    """
    Calcule la différence raster : GWL2 - REF91.

    Paramètres
    ----------
    raster_gwl2 : str
        Raster indice sécheresse GWL2
    raster_ref91 : str
        Raster indice sécheresse REF91
    output_path : str
        Raster de sortie (différence)
    nodata_value : float
        Valeur NoData
    overwrite : bool
        Autoriser l'écrasement
    """

    if os.path.exists(output_path) and not overwrite:
        raise FileExistsError(f"{output_path} existe déjà.")

    with rasterio.open(raster_gwl2) as src_gwl2, rasterio.open(raster_ref91) as src_ref:

        # Vérifications de base
        if src_gwl2.shape != src_ref.shape:
            raise ValueError("Les rasters n'ont pas la même dimension.")
        if src_gwl2.transform != src_ref.transform:
            raise ValueError("Les rasters n'ont pas la même géométrie.")
        if src_gwl2.crs != src_ref.crs:
            raise ValueError("Les rasters n'ont pas le même CRS.")

        profile = src_gwl2.profile.copy()
        profile.update(
            dtype="float32",
            nodata=nodata_value,
            compress="deflate"
        )

        arr_gwl2 = src_gwl2.read(1).astype(np.float32)
        arr_ref = src_ref.read(1).astype(np.float32)

    # Masque nodata
    mask = (
        (arr_gwl2 == nodata_value) |
        (arr_ref == nodata_value) |
        (~np.isfinite(arr_gwl2)) |
        (~np.isfinite(arr_ref))
    )

    diff = np.full(arr_gwl2.shape, nodata_value, dtype=np.float32)
    diff[~mask] = arr_gwl2[~mask] - arr_ref[~mask]

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(diff, 1)

    print(f"✔ Différence calculée : {output_path}")

    return output_path


In [ ]:
compute_raster_difference(
    raster_gwl2="../data/processed/Climat/indices_secheresse_ensemble/I_climat_gwl2_ensemble.tif",
    raster_ref91="../data/processed/Climat/indices_secheresse_ensemble/I_climat_ref91_ensemble.tif",
    output_path="../data/processed/Climat/indices_secheresse_ensemble/I_climat_diff_gwl2_minus_ref91.tif",
    nodata_value=-9999.0,
    overwrite=True
)

In [ ]:
# renormalisation de l'indice de différence
normalize_with_clip(input_path=r"../data/processed/Climat/indices_secheresse_ensemble/I_climat_diff_gwl2_minus_ref91.tif",
                    output_path=r"../data/processed/Climat/indices_secheresse_ensemble/I_climat_diff_gwl2_minus_ref91_normalized.tif",
                    pmin=0.1,pmax=99.9, overwrite=True)

## 9. Pipeline sur les paramètres climatiques séparés

In [ ]:
# fonctions réutilisées
"""def seasonal_precipitation(ds_pr, season):
    
    Précipitations saisonnières (mm/saison)

    pr = ds_pr["pr"].sel(time=season)

    pr.name = f"pr_{season}"
    pr.attrs["description"] = f"Seasonal precipitation ({season})"

    return pr"""
# modification pour assurer la gestion des NoData
def seasonal_precipitation(ds_pr, season, nodata_value=-9999.0):
    """
    Précipitations saisonnières (mm/saison).
    Les pixels dont la somme vaut 0 sont considérés comme NoData.
    """

    pr = ds_pr["pr"].sel(time=season)

    # Sécurité : valeurs finies uniquement
    pr = pr.where(np.isfinite(pr))

    # Remplacement des zéros par NaN (cas artefact NoData)
    pr = pr.where(pr != 0)

    # Optionnel : remplacement explicite par nodata_value
    pr = pr.fillna(nodata_value)

    pr.name = f"pr_{season}"
    pr.attrs.update({
        "description": f"Seasonal precipitation ({season})",
        "units": "mm/season",
        "_FillValue": nodata_value,
        "missing_value": nodata_value
    })

    return pr


def seasonal_temperature(ds_tas, season):
    """
    Température moyenne saisonnière (°C)
    """
    tas = ds_tas["tas"].sel(time=season)

    tas.name = f"tas_{season}"
    tas.attrs["description"] = f"Seasonal mean temperature ({season})"

    return tas


def summer_tasmax(ds_tasmax):
    """
    Température maximale estivale (°C)
    """
    tasmax = ds_tasmax["tasmax"].sel(time="JJA")

    tasmax.name = "tasmax_JJA"
    tasmax.attrs["description"] = "Summer maximum temperature (JJA)"

    return tasmax

def spring_tasmax(ds_tasmax):
    """
    Température maximale printanière (°C)
    """
    tasmax = ds_tasmax["tasmax"].sel(time="MAM")

    tasmax.name = "tasmax_MAM"
    tasmax.attrs["description"] = "Spring maximum temperature (MAM)"

    return tasmax

def spring_tasmin(ds_tasmin):
    """
    Température minimale printanière (°C)
    """
    tasmin = ds_tasmin["tasmin"].sel(time="MAM")

    tasmin.name = "tasmin_MAM"
    tasmin.attrs["description"] = "Spring minimum temperature (MAM)"

    return tasmin

def normalize_climate_variable(da, inverse=False):
    da_n = normalize_minmax(da, inverse=inverse)
    da_n.name = da.name + "_norm"
    da_n.attrs["normalization"] = "min-max spatial"
    return da_n
# Normalisation avancée avec clipping percentile et options
import os
import json
import numpy as np
import rasterio
from rasterio.enums import Resampling

def normalize_with_clip(input_path, output_path, *,
                        pmin=2.0, pmax=98.0,
                        prelog=False,
                        symmetric=False,
                        nodata_keep=True,
                        nodata_value=None,
                        overwrite=False):
    """
    Normalise un raster avec clipping percentile et min-max.
    - symmetric=True : pour variables centrées sur 0 (ex: courbure). On calcule le percentile pmax
      de la distribution des valeurs absolues puis on clip à +/- that_value.
    - prelog=True : applique np.log1p avant clipping (utile pour TWI)
    - pmin,pmax : percentiles (0..100) ; if symmetric only pmax used.
    - nodata_keep : conserve nodata (remplacé par np.nan pendant calcul).
    """
    if os.path.exists(output_path) and not overwrite:
        raise FileExistsError(f"{output_path} exists. Use overwrite=True to replace.")

    with rasterio.open(input_path) as src:
        profile = src.profile.copy()
        arr = src.read(1).astype(np.float32)
        src_nodata = src.nodata if nodata_value is None else nodata_value

    # mask nodata
    if src_nodata is not None:
        mask = (arr == src_nodata) | (~np.isfinite(arr))
    else:
        mask = ~np.isfinite(arr)
    valid = arr[~mask]

    if valid.size == 0:
        raise ValueError("No valid pixels found in input raster.")

    # optionally apply pre-log (on valid values only)
    if prelog:
        valid_proc = np.log1p(valid)
    else:
        valid_proc = valid

    # symmetric clipping around 0 for centered variables
    if symmetric:
        # use percentile on absolute values
        abs_vals = np.abs(valid_proc)
        cutoff = np.percentile(abs_vals, pmax)
        low_val = -cutoff
        high_val = cutoff
        print(f"[symmetric] cutoff (abs) @ p{pmax} = {cutoff:.6g}")
    else:
        low_val = np.percentile(valid_proc, pmin)
        high_val = np.percentile(valid_proc, pmax)
        print(f"percentiles: p{pmin}={low_val:.6g}, p{pmax}={high_val:.6g}")

    # Create working copy (float) and replace nodata with nan
    work = arr.astype(np.float32)
    work[mask] = np.nan

    # if prelog, transform entire working array's valid pixels
    if prelog:
        with np.errstate(invalid='ignore'):
            work = np.where(np.isfinite(work), np.log1p(work), work)

    # clip
    work_clipped = np.clip(work, low_val, high_val)

    # min-max normalize
    denom = (high_val - low_val)
    if denom == 0:
        # constant field after clipping
        norm = np.full_like(work_clipped, np.nan, dtype=np.float32)
        norm[np.isfinite(work_clipped)] = 0.5
    else:
        norm = (work_clipped - low_val) / denom
    norm[mask] = profile.get("nodata", np.nan) if nodata_keep else np.nan

    # update profile
    profile.update(dtype=rasterio.float32, compress='deflate')
    if nodata_keep:
        profile.update(nodata=profile.get("nodata", -9999))
        out_nodata = profile["nodata"]
    else:
        out_nodata = np.nan

    # ensure output dir
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # write raster
    with rasterio.open(output_path, "w", **profile) as dst:
        write_arr = np.where(np.isfinite(norm), norm, out_nodata).astype(np.float32)
        dst.write(write_arr, 1)

    # save metadata about normalization
    meta = {
        "input": os.path.basename(input_path),
        "output": os.path.basename(output_path),
        "pmin": pmin,
        "pmax": pmax,
        "prelog": bool(prelog),
        "symmetric": bool(symmetric),
        "low_val": float(low_val),
        "high_val": float(high_val),
        "nodata_kept": bool(nodata_keep),
    }
    meta_path = os.path.splitext(output_path)[0] + "_meta.json"
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print(f"Saved normalized raster: {output_path}")
    print(f"Saved metadata: {meta_path}")
    return output_path, meta_path


In [ ]:
# pipeline pour les paramètres
import xarray as xr
import numpy as np
def compute_seasonal_climate_variables(
    ds_pr_path,
    ds_tas_path,
    ds_tasmax_path=None,
    ds_tasmin_path=None,
    output_path=None,
    model_name=None,
    gwl=None,
    overwrite=False
):
    """
    Calcule et sauvegarde les variables climatiques saisonnières
    pour un modèle et un horizon donnés.
    """

    ds_pr = xr.open_dataset(ds_pr_path, chunks={"N": 100, "E": 100})
    #for v in ds_pr.data_vars:
    #    ds_pr[v] = ds_pr[v].where(np.isfinite(ds_pr[v]))
    
    ds_tas = xr.open_dataset(ds_tas_path, chunks={"N": 100, "E": 100})
    
    # --- PR ---
    pr_mam = seasonal_precipitation(ds_pr, "MAM")
    pr_jja = seasonal_precipitation(ds_pr, "JJA")

    # --- TAS ---
    tas_mam = seasonal_temperature(ds_tas, "MAM")
    tas_jja = seasonal_temperature(ds_tas, "JJA")

    outputs = {
        "pr_MAM": pr_mam,
        "pr_JJA": pr_jja,
        "tas_MAM": tas_mam,
        "tas_JJA": tas_jja,
    }

    # --- TASMAX ---
    if ds_tasmax_path is not None:
        ds_tasmax = xr.open_dataset(ds_tasmax_path, chunks={"N": 100, "E": 100})
        tasmax_jja = summer_tasmax(ds_tasmax)
        outputs["tasmax_JJA"] = tasmax_jja
        tasmax_mam = spring_tasmax(ds_tasmax)
        outputs["tasmax_MAM"] = tasmax_mam

    # --- TASMIN ---
    if ds_tasmin_path is not None:
        ds_tasmin = xr.open_dataset(ds_tasmin_path, chunks={"N": 100, "E": 100})
        tasmin_mam = spring_tasmin(ds_tasmin)
        outputs["tasmin_MAM"] = tasmin_mam

    # --- SAUVEGARDE ---
    for name, da in outputs.items():
        out_file = (
            f"{output_path}/{name}_{model_name}_{gwl}.nc"
        )
        da.to_netcdf(out_file)
        print(f"✔ Sauvé : {out_file}")

    return outputs


In [ ]:
# pipeline pour tous les modèles
# Noresm
compute_seasonal_climate_variables(
    ds_pr_path="../data/processed/Climat/typical_seasons/noresm_gwl2_pr_seasonal.nc",
    ds_tas_path="../data/processed/Climat/typical_seasons/noresm_gwl2_tas_seasonal.nc",
    ds_tasmax_path="../data/processed/Climat/typical_seasons/noresm_gwl2_tasmax_seasonal.nc",
    output_path="../data/processed/Climat/param_secheresse",
    model_name="noresm",
    gwl="gwl2"
)

compute_seasonal_climate_variables(
    ds_pr_path="../data/processed/Climat/typical_seasons/noresm_ref91_pr_seasonal.nc",
    ds_tas_path="../data/processed/Climat/typical_seasons/noresm_ref91_tas_seasonal.nc",
    ds_tasmax_path="../data/processed/Climat/typical_seasons/noresm_ref91_tasmax_seasonal.nc",
    output_path="../data/processed/Climat/param_secheresse",
    model_name="noresm",
    gwl="ref91"
)
# ECEarth
compute_seasonal_climate_variables(
    ds_pr_path="../data/processed/Climat/typical_seasons/ecearth_gwl2_pr_seasonal.nc",
    ds_tas_path="../data/processed/Climat/typical_seasons/ecearth_gwl2_tas_seasonal.nc",
    ds_tasmax_path="../data/processed/Climat/typical_seasons/ecearth_gwl2_tasmax_seasonal.nc",
    output_path="../data/processed/Climat/param_secheresse",
    model_name="ecearth",
    gwl="gwl2"
)
compute_seasonal_climate_variables(
    ds_pr_path="../data/processed/Climat/typical_seasons/ecearth_ref91_pr_seasonal.nc",
    ds_tas_path="../data/processed/Climat/typical_seasons/ecearth_ref91_tas_seasonal.nc",
    ds_tasmax_path="../data/processed/Climat/typical_seasons/ecearth_ref91_tasmax_seasonal.nc",
    output_path="../data/processed/Climat/param_secheresse",
    model_name="ecearth",
    gwl="ref91"
)
# HadGEM
compute_seasonal_climate_variables(
    ds_pr_path="../data/processed/Climat/typical_seasons/hadgem_gwl2_pr_seasonal.nc",
    ds_tas_path="../data/processed/Climat/typical_seasons/hadgem_gwl2_tas_seasonal.nc",
    ds_tasmax_path="../data/processed/Climat/typical_seasons/hadgem_gwl2_tasmax_seasonal.nc",
    output_path="../data/processed/Climat/param_secheresse",
    model_name="hadgem",
    gwl="gwl2"
)
compute_seasonal_climate_variables(
    ds_pr_path="../data/processed/Climat/typical_seasons/hadgem_ref91_pr_seasonal.nc",
    ds_tas_path="../data/processed/Climat/typical_seasons/hadgem_ref91_tas_seasonal.nc",
    ds_tasmax_path="../data/processed/Climat/typical_seasons/hadgem_ref91_tasmax_seasonal.nc",
    output_path="../data/processed/Climat/param_secheresse",
    model_name="hadgem",
    gwl="ref91"
)


In [ ]:
# Pipeline: conversion en rasters, normalisation, puis agrégation inter-modèles et renormalisation
# On suit ainsi la même méthode que pour les indices multi-paramètres

from pathlib import Path

INPUT_DIR = "../data/processed/Climat/param_secheresse/"
OUTPUT_DIR = "../data/processed/Climat/param_secheresse_tif/"

nc_files = sorted(Path(INPUT_DIR).glob("*.nc"))

for nc_file in nc_files:
    print(f"\nTraitement : {nc_file.name}")
    export_all_indices_nc_to_geotiff(
        nc_path=str(nc_file),
        output_dir=OUTPUT_DIR
    )

In [ ]:
# agrégation inter-modèles des paramètres climatiques saisonniers

# précipitations JJA
# reférence ref91
input_ref91 = [
    "../data/processed/Climat/param_secheresse_tif/pr_JJA_ecearth_ref91.tif",
    "../data/processed/Climat/param_secheresse_tif/pr_JJA_hadgem_ref91.tif",
    "../data/processed/Climat/param_secheresse_tif/pr_JJA_noresm_ref91.tif",
]

aggregate_rasters_mean(
    input_ref91,
    "../data/processed/Climat/param_secheresse_ensemble/pr_JJA_ensemble_ref91.tif",
    overwrite=True
)
# GWL2
input_gwl2 = [
    "../data/processed/Climat/param_secheresse_tif/pr_JJA_ecearth_gwl2.tif",
    "../data/processed/Climat/param_secheresse_tif/pr_JJA_hadgem_gwl2.tif",
    "../data/processed/Climat/param_secheresse_tif/pr_JJA_noresm_gwl2.tif",
]

aggregate_rasters_mean(
    input_gwl2,
    "../data/processed/Climat/param_secheresse_ensemble/pr_JJA_ensemble_gwl2.tif",
    overwrite=True
)
# précipitations MAM
# reférence ref91
input_ref91 = [
    "../data/processed/Climat/param_secheresse_tif/pr_MAM_ecearth_ref91.tif",
    "../data/processed/Climat/param_secheresse_tif/pr_MAM_hadgem_ref91.tif",
    "../data/processed/Climat/param_secheresse_tif/pr_MAM_noresm_ref91.tif",
]

aggregate_rasters_mean(
    input_ref91,
    "../data/processed/Climat/param_secheresse_ensemble/pr_MAM_ensemble_ref91.tif",
    overwrite=True
)
# GWL2
input_gwl2 = [
    "../data/processed/Climat/param_secheresse_tif/pr_MAM_ecearth_gwl2.tif",
    "../data/processed/Climat/param_secheresse_tif/pr_MAM_hadgem_gwl2.tif",
    "../data/processed/Climat/param_secheresse_tif/pr_MAM_noresm_gwl2.tif",
]
aggregate_rasters_mean(
    input_gwl2,
    "../data/processed/Climat/param_secheresse_ensemble/pr_MAM_ensemble_gwl2.tif",
    overwrite=True
)
# température tas JJA
# reférence ref91
input_ref91 = [
    "../data/processed/Climat/param_secheresse_tif/tas_JJA_ecearth_ref91.tif",
    "../data/processed/Climat/param_secheresse_tif/tas_JJA_hadgem_ref91.tif",
    "../data/processed/Climat/param_secheresse_tif/tas_JJA_noresm_ref91.tif",
]
aggregate_rasters_mean(
    input_ref91,
    "../data/processed/Climat/param_secheresse_ensemble/tas_JJA_ensemble_ref91.tif",
    overwrite=True
)
# GWL2
input_gwl2 = [
    "../data/processed/Climat/param_secheresse_tif/tas_JJA_ecearth_gwl2.tif",
    "../data/processed/Climat/param_secheresse_tif/tas_JJA_hadgem_gwl2.tif",
    "../data/processed/Climat/param_secheresse_tif/tas_JJA_noresm_gwl2.tif",
]
aggregate_rasters_mean(
    input_gwl2,
    "../data/processed/Climat/param_secheresse_ensemble/tas_JJA_ensemble_gwl2.tif",
    overwrite=True
)
# température tas MAM
# reférence ref91
input_ref91 = [
    "../data/processed/Climat/param_secheresse_tif/tas_MAM_ecearth_ref91.tif",
    "../data/processed/Climat/param_secheresse_tif/tas_MAM_hadgem_ref91.tif",
    "../data/processed/Climat/param_secheresse_tif/tas_MAM_noresm_ref91.tif",
]
aggregate_rasters_mean(
    input_ref91,
    "../data/processed/Climat/param_secheresse_ensemble/tas_MAM_ensemble_ref91.tif",
    overwrite=True
)
# GWL2
input_gwl2 = [
    "../data/processed/Climat/param_secheresse_tif/tas_MAM_ecearth_gwl2.tif",
    "../data/processed/Climat/param_secheresse_tif/tas_MAM_hadgem_gwl2.tif",
    "../data/processed/Climat/param_secheresse_tif/tas_MAM_noresm_gwl2.tif",
]
aggregate_rasters_mean(
    input_gwl2,
    "../data/processed/Climat/param_secheresse_ensemble/tas_MAM_ensemble_gwl2.tif",
    overwrite=True
)
# température tasmax JJA
# reférence ref91
input_ref91 = [
    "../data/processed/Climat/param_secheresse_tif/tasmax_JJA_ecearth_ref91.tif",
    "../data/processed/Climat/param_secheresse_tif/tasmax_JJA_hadgem_ref91.tif",
    "../data/processed/Climat/param_secheresse_tif/tasmax_JJA_noresm_ref91.tif",
]
aggregate_rasters_mean(
    input_ref91,
    "../data/processed/Climat/param_secheresse_ensemble/tasmax_JJA_ensemble_ref91.tif",
    overwrite=True
)
# GWL2
input_gwl2 = [
    "../data/processed/Climat/param_secheresse_tif/tasmax_JJA_ecearth_gwl2.tif",
    "../data/processed/Climat/param_secheresse_tif/tasmax_JJA_hadgem_gwl2.tif",
    "../data/processed/Climat/param_secheresse_tif/tasmax_JJA_noresm_gwl2.tif",
]
aggregate_rasters_mean(
    input_gwl2,
    "../data/processed/Climat/param_secheresse_ensemble/tasmax_JJA_ensemble_gwl2.tif",
    overwrite=True
)
# température tasmax MAM
# reférence ref91
input_ref91 = [
    "../data/processed/Climat/param_secheresse_tif/tasmax_MAM_ecearth_ref91.tif",
    "../data/processed/Climat/param_secheresse_tif/tasmax_MAM_hadgem_ref91.tif",
    "../data/processed/Climat/param_secheresse_tif/tasmax_MAM_noresm_ref91.tif",
]
aggregate_rasters_mean(
    input_ref91,
    "../data/processed/Climat/param_secheresse_ensemble/tasmax_MAM_ensemble_ref91.tif",
    overwrite=True
)
# GWL2
input_gwl2 = [
    "../data/processed/Climat/param_secheresse_tif/tasmax_MAM_ecearth_gwl2.tif",
    "../data/processed/Climat/param_secheresse_tif/tasmax_MAM_hadgem_gwl2.tif",
    "../data/processed/Climat/param_secheresse_tif/tasmax_MAM_noresm_gwl2.tif",
]
aggregate_rasters_mean(
    input_gwl2,
    "../data/processed/Climat/param_secheresse_ensemble/tasmax_MAM_ensemble_gwl2.tif",
    overwrite=True
)

In [ ]:
# différence entre GWL et Ref91 pour les paramètres climatiques saisonniers
# precipitations JJA
compute_raster_difference(
    raster_gwl2="../data/processed/Climat/param_secheresse_ensemble/pr_JJA_ensemble_gwl2.tif",
    raster_ref91="../data/processed/Climat/param_secheresse_ensemble/pr_JJA_ensemble_ref91.tif",
    output_path="../data/processed/Climat/param_secheresse_ensemble/pr_JJA_diff_gwl2_minus_ref91.tif",
    nodata_value=-9999.0,
    overwrite=True
)
# precipitations MAM
compute_raster_difference(
    raster_gwl2="../data/processed/Climat/param_secheresse_ensemble/pr_MAM_ensemble_gwl2.tif",
    raster_ref91="../data/processed/Climat/param_secheresse_ensemble/pr_MAM_ensemble_ref91.tif",
    output_path="../data/processed/Climat/param_secheresse_ensemble/pr_MAM_diff_gwl2_minus_ref91.tif",
    nodata_value=-9999.0,
    overwrite=True
)
# température tas JJA
compute_raster_difference(
    raster_gwl2="../data/processed/Climat/param_secheresse_ensemble/tas_JJA_ensemble_gwl2.tif",
    raster_ref91="../data/processed/Climat/param_secheresse_ensemble/tas_JJA_ensemble_ref91.tif",
    output_path="../data/processed/Climat/param_secheresse_ensemble/tas_JJA_diff_gwl2_minus_ref91.tif",
    nodata_value=-9999.0,
    overwrite=True
)
# température tas MAM
compute_raster_difference(
    raster_gwl2="../data/processed/Climat/param_secheresse_ensemble/tas_MAM_ensemble_gwl2.tif",
    raster_ref91="../data/processed/Climat/param_secheresse_ensemble/tas_MAM_ensemble_ref91.tif",
    output_path="../data/processed/Climat/param_secheresse_ensemble/tas_MAM_diff_gwl2_minus_ref91.tif",
    nodata_value=-9999.0,
    overwrite=True
)
# température tasmax JJA
compute_raster_difference(
    raster_gwl2="../data/processed/Climat/param_secheresse_ensemble/tasmax_JJA_ensemble_gwl2.tif",
    raster_ref91="../data/processed/Climat/param_secheresse_ensemble/tasmax_JJA_ensemble_ref91.tif",
    output_path="../data/processed/Climat/param_secheresse_ensemble/tasmax_JJA_diff_gwl2_minus_ref91.tif",
    nodata_value=-9999.0,
    overwrite=True
)
# température tasmax MAM
compute_raster_difference(
    raster_gwl2="../data/processed/Climat/param_secheresse_ensemble/tasmax_MAM_ensemble_gwl2.tif",
    raster_ref91="../data/processed/Climat/param_secheresse_ensemble/tasmax_MAM_ensemble_ref91.tif",
    output_path="../data/processed/Climat/param_secheresse_ensemble/tasmax_MAM_diff_gwl2_minus_ref91.tif",
    nodata_value=-9999.0,
    overwrite=True
)
